## 0.1 Init del ambiente (VM de Google Cloud o Colab)

Esta celda deja el ambiente listo y **funciona igual en los dos entornos**:

- **VM de Google Cloud**: la instalacion de la catedra ya monta el bucket en `~/buckets/b1`, no hay nada que montar.
- **Google Colab**: monta el Google Drive y lo enlaza en `/content/buckets/b1`.

Ademas crea las carpetas que usa el pipe completo, instala `kaggle.json` si esta en el bucket
y baja los datasets crudos que falten. Es idempotente: se puede volver a correr sin efectos.

In [1]:
# ── INIT DEL AMBIENTE ─────────────────────────────────────────────────────
# Portable: VM de Google Cloud (bucket ya montado en ~/buckets/b1) y Google Colab.
# No usa %%shell (magic exclusiva de Colab) ni rutas /content fijas.
import os
import shutil
import urllib.request
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/.drive")
    destino = Path("/content/.drive/My Drive/labo3")
    destino.mkdir(parents=True, exist_ok=True)
    Path("/content/buckets").mkdir(parents=True, exist_ok=True)
    BUCKET = Path("/content/buckets/b1")
    if BUCKET.is_symlink():
        BUCKET.unlink()
    if not BUCKET.exists():
        BUCKET.symlink_to(destino)
    print("Entorno: Google Colab")
else:
    # VM de Google Cloud. El bucket lo monta la instalacion de la catedra.
    BUCKET = Path.home() / "buckets" / "b1"
    if not BUCKET.is_dir():
        raise RuntimeError(
            f"No encuentro el bucket en {BUCKET}. Si estas en la VM de Google Cloud, "
            "revisa que el bucket este montado (ls ~/buckets/b1). Si corres en otro "
            "lado, defini LABO3_BUCKET, ej: os.environ['LABO3_BUCKET'] = '/ruta/al/bucket'"
        )
    print("Entorno: VM de Google Cloud")

# Carpetas que usa el pipe completo (01 -> 02 -> 03 -> 04)
for sub in ("datasets", "datasets/preprocesado", "datasets_fe", "exp"):
    (BUCKET / sub).mkdir(parents=True, exist_ok=True)

# Credenciales de Kaggle (solo hacen falta para submitear a la competencia)
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
if kaggle_dst.exists():
    print("kaggle.json: ya estaba instalado")
else:
    for cand in (BUCKET / "kaggle" / "kaggle.json", BUCKET / "kaggle.json"):
        if cand.exists():
            kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"kaggle.json: instalado desde {cand}")
            break
    else:
        print("kaggle.json: NO encontrado en el bucket (solo hace falta para submitear)")

# Descarga de los datasets crudos que falten
URL_ORIGEN = "https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
ARCHIVOS = (
    "sell-in.txt.gz",
    "tb_productos.txt",
    "tb_stocks.txt",
    "product_id_apredecir201912.txt",
)

for archivo in ARCHIVOS:
    destino_archivo = BUCKET / "datasets" / archivo
    if destino_archivo.exists() and destino_archivo.stat().st_size > 0:
        print(f"dataset ya estaba : {archivo}")
        continue
    # Bajamos a .parcial y recien al terminar renombramos: si la spot VM muere
    # a mitad de la descarga, no queda un archivo truncado que parezca completo.
    parcial = destino_archivo.with_suffix(destino_archivo.suffix + ".parcial")
    print(f"bajando           : {archivo} ...")
    urllib.request.urlretrieve(URL_ORIGEN + archivo, parcial)
    parcial.replace(destino_archivo)
    mb = destino_archivo.stat().st_size / 1024**2
    print(f"                    listo ({mb:.1f} MB)")

print(f"\nBUCKET listo en: {BUCKET}")


Entorno: VM de Google Cloud
kaggle.json: ya estaba instalado
dataset ya estaba : sell-in.txt.gz
dataset ya estaba : tb_productos.txt
dataset ya estaba : tb_stocks.txt
dataset ya estaba : product_id_apredecir201912.txt

BUCKET listo en: /home/ds/buckets/b1


## 1-Imports

In [2]:
import logging
import os
from dataclasses import dataclass, field
from pathlib import Path
from time import time
from typing import Literal, Tuple, Optional

import numpy as np
import polars as pl
import sys # Importar sys para dirigir el logging a stdout

# Setup de Logging Industrial
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)], # Dirigir el output del logger a sys.stdout
)
logger = logging.getLogger("Pipeline_Preprocessing")

In [3]:
# ── RUTAS DEL BUCKET ──────────────────────────────────────────────────────
# Unico lugar del notebook donde se definen rutas. El bucket se autodetecta:
#   VM de la catedra -> ~/buckets/b1   |   Colab -> /content/buckets/b1
# Estructura que usa el pipe completo:
#   {BUCKET}/datasets/                  crudos (sell-in.txt.gz, tb_productos.txt, ...)
#   {BUCKET}/datasets/preprocesado/     salida de 01_Preprocesamiento  <-- lo escribe este notebook
#   {BUCKET}/datasets_fe/               salida de 02_FE
#   {BUCKET}/exp/<experimento>/         salida de 03_Optuna
def resolver_bucket() -> Path:
    # 1) LABO3_BUCKET: para correr fuera de la nube (server propio, notebook local).
    #    export LABO3_BUCKET=/ruta/a/mi/bucket   (o os.environ[...] antes de importar)
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    # 2) rutas conocidas: Colab y la VM de GCP
    # ~/buckets/b1 primero: es donde lo monta la instalacion de la catedra,
    # y sirve para cualquier usuario de la VM.
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Opciones: "
        "(a) Colab / VM de GCP -> corre la celda de init del ambiente; "
        "(b) server propio o local -> defini LABO3_BUCKET antes de esta celda, ej. "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"                 # crudos, tal como los baja la celda de init
DIR_PREP = BUCKET / "datasets" / "preprocesado"  # salida de este notebook
DIR_PREP.mkdir(parents=True, exist_ok=True)

logger.info(f"BUCKET   : {BUCKET}")
logger.info(f"crudos   : {DIR_RAW}")
logger.info(f"salida   : {DIR_PREP}")


2026-07-28 23:51:20,184 [INFO] BUCKET   : /home/ds/buckets/b1
2026-07-28 23:51:20,185 [INFO] crudos   : /home/ds/buckets/b1/datasets
2026-07-28 23:51:20,186 [INFO] salida   : /home/ds/buckets/b1/datasets/preprocesado


## 2-Configuración del Experimento (DataClass Centralizado)

In [4]:
@dataclass(frozen=True)
class PreprocessingConfig:
    """Clase inmutable para la parametrización absoluta de experimentos."""

    # Rutas de Ingesta (Compatibles con Google Colab / Local)
    base_dir: Path = DIR_RAW
    sell_in_path: Path = DIR_RAW / "sell-in.txt.gz"
    productos_path: Path = DIR_RAW / "tb_productos.txt"
    apredecir_path: Path = DIR_RAW / "product_id_apredecir201912.txt"
    output_dir: Path = DIR_PREP

    # [Experimento 1] Modo de Agrupamiento:
    # 'A': por Cliente-Producto-Mes | 'B': por Producto-Mes
    group_mode: Literal["A", "B"] = "A"
    default_customer_id: int = 0

    # [Experimento 2] Modo de completar faltantes (Post-Densificación)
    # 'zero': Completa con 0.0 e imita el comportamiento físico de no-venta
    # 'null': Preserva valores NA nativos
    missing_strategy: Literal["zero", "null"] = "zero"

    # [Experimento 3] Modo de Densificación Temporal
    # 'full': Malla cruzada total desde el inicio al fin del tiempo general
    # 'lifecycle': Respeta la vida comercial de cada producto individual
    densify_strategy: Literal["full", "lifecycle"] = "lifecycle"

    # Límites cronológicos del Dataset Histórico Observado
    timeline_start: str = "2017-01-01"
    timeline_end: str = "2019-12-01"

    # Filtro opcional: Mantener solo productos requeridos para el Target final de Negocio (los 780 que hay que predecir)
    filter_target_products_only: bool = True

    # [Muestreo] Quedarse solo con los N productos de mayor volumen (tn total).
    # None = todos. Sirve para validar el pipe entero en minutos y con poca RAM:
    # el subconjunto queda anotado en el nombre del archivo (_smplN), asi que NO
    # pisa al dataset completo y 02/03 lo distinguen solos.
    # La seleccion es deterministica (top-N por tn), no aleatoria -> reproducible.
    sample_n_products: Optional[int] = None

    def get_output_filename(self) -> str:
        """Genera de forma determinista el nombre del archivo Parquet resultante."""
        grp = "grpClienteProducto" if self.group_mode == "A" else "grpProducto"
        missing = "fill0" if self.missing_strategy == "zero" else "fillNA"
        dense = "denseFull" if self.densify_strategy == "full" else "denseLife"
        tgt = "_tgtFilter" if self.filter_target_products_only else ""
        smpl = f"_smpl{self.sample_n_products}" if self.sample_n_products else ""
        return f"preprocesado_{grp}_{missing}_{dense}{tgt}{smpl}.parquet"

## 3 - Validaciones de Datos Automáticas

In [5]:
class PreprocessingValidationError(Exception):
    """Excepción de control para detener el pipeline ante fallos críticos de sanidad."""
    pass


def validate_input_integrity(df_sell_in: pl.DataFrame, df_prod: pl.DataFrame) -> None:
    """Valida la integridad referencial y estructural de los insumos."""
    # Validación de duplicados en catálogo
    if df_prod.select("product_id").is_duplicated().any():
        raise PreprocessingValidationError("Existen llaves 'product_id' duplicadas en el maestro de productos.")

    # Identificar huérfanos sin detener, enviando advertencia controlada al log
    orphans = df_sell_in.select("product_id").unique().join(
        df_prod.select("product_id").unique(), on="product_id", how="left_anti"
    )
    if not orphans.is_empty():
        logger.warning(f"Calidad Alerta: Existen {orphans.height} product_ids transaccionados que faltan en el catálogo.")

## 4 - Desacoplamiento de Lectura

In [6]:
def read_and_clean_sources(config: PreprocessingConfig) -> Tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:
    """Carga los archivos en modo diferido (Lazy), unificando los tipos de datos temporales."""
    logger.info("Iniciando fase de lectura de fuentes transaccionales...")

    # Lectura diferida con delimitador correcto
    lf_sell_in = pl.scan_csv(config.sell_in_path, separator="\t")
    lf_productos = pl.scan_csv(config.productos_path, separator="\t")
    lf_apredecir = pl.scan_csv(config.apredecir_path, separator="\t")

    # Estandarización del eje temporal de Periodo (int YYYYMM -> pl.Date)
    lf_sell_in = lf_sell_in.with_columns(
        pl.format("{}-01", pl.col("periodo"))
        .str.to_date("%Y%m-01")
        .alias("periodo")
    )

    return lf_sell_in, lf_productos, lf_apredecir

## 5 - Transformaciones y Lógica del Experimento

In [7]:
def apply_aggregation_mode(lf: pl.LazyFrame, config: PreprocessingConfig) -> pl.LazyFrame:
    """[Experimento 1]: Agrupación de Demanda y Generación Consistente del Agrupacion_ID."""
    logger.info(f"Ejecutando agregación espacial en Modo: {config.group_mode}")

    # Lista base de métricas numéricas agregables
    metrics = ["tn", "cust_request_qty", "cust_request_tn", "plan_precios_cuidados"]
    agg_exprs = [pl.col(m).sum().alias(m) for m in metrics]

    if config.group_mode == "A":
        # Alternativa A: Granularidad Fina de Cliente
        lf_agg = lf.group_by(["periodo", "customer_id", "product_id"]).agg(agg_exprs)
    elif config.group_mode == "B":
        # Alternativa B: Desestimar clientes, seteando la convención de Customer_ID = 0
        lf_agg = lf.group_by(["periodo", "product_id"]).agg(agg_exprs).with_columns(
            pl.lit(config.default_customer_id).alias("customer_id")
        )
    else:
        raise ValueError(f"Modo de grupo inválido: {config.group_mode}")

    # Firma obligatoria para consistencia de Series Temporales aguas abajo
    lf_agg = lf_agg.with_columns(
        (pl.col("customer_id").cast(pl.Int64) * 100000 + pl.col("product_id").cast(pl.Int64)).alias("Agrupacion_ID")
    )
    return lf_agg


def apply_densification_strategy(lf: pl.LazyFrame, config: PreprocessingConfig) -> pl.LazyFrame:
    """[Experimento 3]: Algoritmo de densificación temporal avanzada."""
    logger.info(f"Construyendo rejilla temporal bajo estrategia: {config.densify_strategy}")

    start_dt = pl.lit(config.timeline_start).str.to_date()
    end_dt = pl.lit(config.timeline_end).str.to_date()

    # Universo único observado de agrupaciones combinatorias espaciales
    unique_series = lf.select(["Agrupacion_ID", "customer_id", "product_id"]).unique()

    if config.densify_strategy == "full":
        # Estrategia A: Producto Cartesiano Completo
        # Crear un LazyFrame a partir de la serie de fechas
        period_series_lf = pl.LazyFrame({"periodo": pl.date_range(start_dt, end_dt, interval="1mo", eager=True)}).lazy()
        grid = unique_series.join(
            period_series_lf,
            how="cross"
        )
    elif config.densify_strategy == "lifecycle":
        # Estrategia B: Delimitación paramétrica del ciclo de vida del SKU
        lifecycle_bounds = lf.group_by("product_id").agg([
            pl.col("periodo").min().alias("birth_date"),
            pl.col("periodo").max().alias("death_date")
        ])

        # Restricciones estrictas de frontera: No asumir muerte/nacimiento en los extremos del dataset
        lifecycle_bounds = lifecycle_bounds.with_columns([
            pl.when(pl.col("birth_date") == start_dt).then(start_dt).otherwise(pl.col("birth_date")).alias("birth_date"),
            pl.when(pl.col("death_date") == end_dt).then(end_dt).otherwise(pl.col("death_date")).alias("death_date")
        ])

        # Explotar la ventana de tiempo válida exclusiva de cada producto
        grid = unique_series.join(lifecycle_bounds, on="product_id", how="inner")
        grid = grid.with_columns(
            pl.date_ranges(pl.col("birth_date"), pl.col("death_date"), interval="1mo").alias("periodo")
        ).explode("periodo").drop(["birth_date", "death_date"])

    # Unión izquierda para rellenar los valores observados en la rejilla generada
    dense_lf = grid.join(lf, on=["Agrupacion_ID", "customer_id", "product_id", "periodo"], how="left")
    return dense_lf


def apply_imputation_strategy(lf: pl.LazyFrame, config: PreprocessingConfig) -> pl.LazyFrame:
    """[Experimento 2]: Tratamiento paramétrico de celdas vacías post-densificación."""
    logger.info(f"Tratando nulos resultantes con estrategia: {config.missing_strategy}")

    if config.missing_strategy == "zero":
        lf = lf.with_columns([
            pl.col("tn").fill_null(0.0),
            pl.col("cust_request_qty").fill_null(0),
            pl.col("cust_request_tn").fill_null(0.0),
            pl.col("plan_precios_cuidados").fill_null(0)
        ])
    elif config.missing_strategy == "null":
        # Mantener nulos nativos para procesamiento avanzado posterior o imputación interna de LightGBM
        pass
    return lf

## 6 - Orquestador del Pipeline y Exportación

In [8]:
def run_pipeline(config: PreprocessingConfig) -> Path:
    """Orquestador maestro optimizado para cómputo por flujos (Streaming Execution)."""
    t_start = time()
    config.output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Ingesta diferida
    lf_sell_in, lf_productos, lf_apredecir = read_and_clean_sources(config)

    # Filtro opcional preventivo para acotar el experimento a las demandas objetivo (los 780 productos que hay que predecir)
    if config.filter_target_products_only:
        target_ids = lf_apredecir.select("product_id").unique()
        lf_sell_in = lf_sell_in.join(target_ids, on="product_id", how="inner")

    # Muestreo opcional: los N productos de mayor volumen. Se aplica DESPUES del
    # filtro de target, asi el subconjunto sale de los productos que ya quedaron.
    if config.sample_n_products is not None:
        top_ids = (
            lf_sell_in.group_by("product_id")
            .agg(pl.col("tn").sum().alias("_tn_total"))
            .sort("_tn_total", descending=True)
            .head(config.sample_n_products)
            .select("product_id")
        )
        lf_sell_in = lf_sell_in.join(top_ids, on="product_id", how="inner")
        logger.info(f"MUESTREO ACTIVO: solo los {config.sample_n_products} productos de mayor tn")

    # 2. Encadenamiento del grafo de ejecución lógica (Lazy Execution)
    processed_graph = (
        lf_sell_in
        .pipe(apply_aggregation_mode, config=config)
        .pipe(apply_densification_strategy, config=config)
        .pipe(apply_imputation_strategy, config=config)
    )

    # Pegar catálogo de productos antes de la salida definitiva
    processed_graph = processed_graph.join(lf_productos, on="product_id", how="left")

    # Mostrar dimensiones antes de guardar
    logger.info(f"Dimensiones del dataset procesado (filas, columnas) antes de guardar: {processed_graph.collect().shape}")

    # 3. Compilación Física de Resultados usando Motor Streaming de Polars
    out_file = config.output_dir / config.get_output_filename()
    logger.info(f"Compilando grafo y volcando datos en formato Parquet hacia: {out_file}")

    # .sink_parquet procesa de forma eficiente sin cargar todo el dataset simultáneamente en RAM
    processed_graph.sink_parquet(out_file)

    # 4. Validaciones Finales Ansiosas (Eager Verification)
    df_final = pl.read_parquet(out_file)

    # Validación de unicidad de llave primaria de series de tiempo
    is_duplicated = df_final.select(["Agrupacion_ID", "periodo"]).is_duplicated().any()
    if is_duplicated:
        raise PreprocessingValidationError("Error de Consistencia: Se generaron llaves duplicadas (Agrupacion_ID, periodo).")

    # 5. Métricas de Control Informativo (Logging)
    duration = time() - t_start
    logger.info("==================================================")
    logger.info(f" PROCESAMIENTO COMPLETADO EXITOSAMENTE")
    logger.info(f" Archivo generado: {config.get_output_filename()}")
    logger.info(f" Filas finales registradas: {df_final.height}")
    logger.info(f" Series temporales únicas: {df_final['Agrupacion_ID'].n_unique()}")
    logger.info(f" Tiempo consumido por el Pipeline: {duration:.2f} segundos")
    logger.info("==================================================")

    return out_file # Retornar la ruta del archivo generado

## 7 - Desencadenamiento del Experimento Activo

In [9]:
if __name__ == "__main__":
    # Desde aquí el experimentador puede conmutar las variables de control
    # y lanzar ejecuciones masivas en paralelo modificando esta estructura.

    config_experimento_1 = PreprocessingConfig(
        # Modo de Agrupamiento: 'A': por Cliente-Producto-Mes | 'B': por Producto-Mes
        group_mode="A",

        #Modo de completar faltantes (Post-Densificación)
        # 'zero': Completa con 0.0 e imita el comportamiento físico de no-venta
        # 'null': Preserva valores NA nativos
        missing_strategy="zero",

        # Modo de Densificación Temporal
        # 'full': Malla cruzada total desde el inicio al fin del tiempo general
        # 'lifecycle': Respeta la vida comercial de cada producto individual
        densify_strategy="lifecycle", # 'full' o 'lifecycle'


        # Filtrar solo productos requeridos para el Target final de Negocio (los 780 que hay que predecir)
        filter_target_products_only=False
    )

    last_generated_file_path = run_pipeline(config_experimento_1)


2026-07-28 23:51:20,247 [INFO] Iniciando fase de lectura de fuentes transaccionales...
2026-07-28 23:51:20,575 [INFO] Ejecutando agregación espacial en Modo: A
2026-07-28 23:51:20,618 [INFO] Construyendo rejilla temporal bajo estrategia: lifecycle
2026-07-28 23:51:20,702 [INFO] Tratando nulos resultantes con estrategia: zero


/tmp/ipykernel_13219/3452575313.py:62: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  ).explode("periodo").drop(["birth_date", "death_date"])


2026-07-28 23:51:24,191 [INFO] Dimensiones del dataset procesado (filas, columnas) antes de guardar: (10792694, 14)
2026-07-28 23:51:24,192 [INFO] Compilando grafo y volcando datos en formato Parquet hacia: /home/ds/buckets/b1/datasets/preprocesado/preprocesado_grpClienteProducto_fill0_denseLife.parquet
2026-07-28 23:51:28,813 [INFO] ==================================================
2026-07-28 23:51:28,814 [INFO]  PROCESAMIENTO COMPLETADO EXITOSAMENTE
2026-07-28 23:51:28,814 [INFO]  Archivo generado: preprocesado_grpClienteProducto_fill0_denseLife.parquet
2026-07-28 23:51:28,815 [INFO]  Filas finales registradas: 10792694
2026-07-28 23:51:28,898 [INFO]  Series temporales únicas: 374439
2026-07-28 23:51:28,898 [INFO]  Tiempo consumido por el Pipeline: 8.57 segundos
2026-07-28 23:51:28,899 [INFO] ==================================================


In [10]:
config_experimento_2 = PreprocessingConfig(
    group_mode="B",             # Agrupar solo por Producto-Mes
    missing_strategy="null",    # Preservar valores NA nativos
    densify_strategy="full",    # Malla cruzada total de tiempo
    filter_target_products_only=True #(Solo los 780 productos que hay que predecir)
)

last_generated_file_path = run_pipeline(config_experimento_2)

2026-07-28 23:51:28,908 [INFO] Iniciando fase de lectura de fuentes transaccionales...
2026-07-28 23:51:28,909 [INFO] Ejecutando agregación espacial en Modo: B
2026-07-28 23:51:28,910 [INFO] Construyendo rejilla temporal bajo estrategia: full
2026-07-28 23:51:28,914 [INFO] Tratando nulos resultantes con estrategia: null
2026-07-28 23:51:29,205 [INFO] Dimensiones del dataset procesado (filas, columnas) antes de guardar: (28080, 14)
2026-07-28 23:51:29,207 [INFO] Compilando grafo y volcando datos en formato Parquet hacia: /home/ds/buckets/b1/datasets/preprocesado/preprocesado_grpProducto_fillNA_denseFull_tgtFilter.parquet
2026-07-28 23:51:29,721 [INFO] ==================================================
2026-07-28 23:51:29,721 [INFO]  PROCESAMIENTO COMPLETADO EXITOSAMENTE
2026-07-28 23:51:29,722 [INFO]  Archivo generado: preprocesado_grpProducto_fillNA_denseFull_tgtFilter.parquet
2026-07-28 23:51:29,723 [INFO]  Filas finales registradas: 28080
2026-07-28 23:51:29,724 [INFO]  Series tempor

## 8 - Documentación para la Siguiente Etapa del Pipeline

### Formato de Archivos de Salida del Preprocesamiento

Los archivos generados por este pipeline de preprocesamiento se encuentran en formato Parquet y se almacenan en el directorio especificado por `PreprocessingConfig.output_dir`. El nombre de cada archivo Parquet sigue una convención determinista basada en los parámetros del experimento:

`preprocesado_{group_mode}_{missing_strategy}_{densify_strategy}{filter_target_products_only}.parquet`

Donde:

-   `{group_mode}`: Indica la estrategia de agrupamiento:
    -   `grpClienteProducto`: Si `group_mode` fue 'A' (por Cliente-Producto-Mes).
    -   `grpProducto`: Si `group_mode` fue 'B' (por Producto-Mes).

-   `{missing_strategy}`: Indica cómo se trataron los valores faltantes post-densificación:
    -   `fill0`: Si `missing_strategy` fue 'zero' (completado con 0.0).
    -   `fillNA`: Si `missing_strategy` fue 'null' (valores NA preservados).

-   `{densify_strategy}`: Indica la estrategia de densificación temporal:
    -   `denseFull`: Si `densify_strategy` fue 'full' (malla cruzada completa).
    -   `denseLife`: Si `densify_strategy` fue 'lifecycle' (vida comercial de cada producto).

-   `{filter_target_products_only}`: Indica si se aplicó el filtro por productos objetivo:
    -   `_tgtFilter`: Si `filter_target_products_only` fue `True`.
    -   (vacío): Si `filter_target_products_only` fue `False`.

**Ejemplo de Nombre de Archivo:**

`preprocesado_grpClienteProducto_fill0_denseLife_tgtFilter.parquet`

Este archivo contendrá series de tiempo densificadas a nivel Cliente-Producto-Mes, con nulos llenados con ceros, respetando el ciclo de vida de los productos y filtrando solo los productos objetivo.

### Estructura de Datos del Archivo Parquet

El DataFrame resultante en cada archivo Parquet incluirá las siguientes columnas esenciales para la siguiente etapa (modelado o análisis):

-   **`periodo`**: Tipo `Date`, representa el inicio del mes (YYYY-MM-01) de cada observación.
-   **`customer_id`**: Tipo `Int64`, identificador único del cliente. Será 0 si `group_mode` es 'B'.
-   **`product_id`**: Tipo `Int64`, identificador único del producto.
-   **`Agrupacion_ID`**: Tipo `Int64`, clave primaria compuesta por `(customer_id * 100000 + product_id)` para identificar unívocamente cada serie temporal.
-   **`tn`**: Tipo `Float64`, toneladas de venta registradas (o 0.0 si se imputó).
-   **`cust_request_qty`**: Tipo `Int64`, cantidad solicitada por el cliente (o 0 si se imputó).
-   **`cust_request_tn`**: Tipo `Float64`, toneladas solicitadas por el cliente (o 0.0 si se imputó).
-   **`plan_precios_cuidados`**: Tipo `Int64`, indicador de si el producto estaba bajo plan de precios cuidados (o 0 si se imputó).
-   **Columnas del maestro de `tb_productos`**: Todas las columnas del archivo `tb_productos.txt` estarán unidas (`how='left'`) a las series de tiempo, proveyendo metadatos del producto (e.g., `brand`, `category`, etc.).

### Consideraciones para la Siguiente Etapa

1.  **Unicidad de Series:** La combinación `(Agrupacion_ID, periodo)` garantiza una clave primaria única para cada observación de serie temporal.
2.  **Manejo de Nulos:** La estrategia de `missing_strategy` debe ser considerada en la etapa de modelado. Si se eligió 'null', los modelos de ML deberán ser capaces de manejar nulos internamente o requerirán una imputación adicional.
3.  **Filtrado por Productos Objetivo:** Si `filter_target_products_only` fue `True`, el dataset ya estará acotado a los `product_id` presentes en `product_id_apredecir201912.txt`.
4.  **Uso de Polars:** Se recomienda continuar utilizando Polars para la carga y manipulación de estos archivos Parquet debido a su eficiencia en memoria y velocidad, especialmente para grandes volúmenes de datos.

## 9 - Verificación del Archivo de Salida

In [11]:
import polars as pl

# Usamos la ruta del archivo generada por la última ejecución del pipeline.
# Esta variable 'last_generated_file_path' se habrá establecido en la celda anterior.
output_file_path = last_generated_file_path

logger.info(f"Cargando archivo: {output_file_path}")

try:
    df_output = pl.read_parquet(output_file_path)

    print(f"\n--- Dimensiones del DataFrame Final ---")
    print(f"Filas: {df_output.height}")
    print(f"Columnas: {df_output.width}")

    print(f"\n--- Primeros 5 Registros ---")
    display(df_output.head(5))

    print(f"\n--- Nombres de los Campos (Columnas) ---")
    for col_name in df_output.columns:
        print(col_name)

except Exception as e:
    logger.error(f"No se pudo cargar o inspeccionar el archivo Parquet: {e}")
    print("Asegúrate de haber ejecutado el pipeline de preprocesamiento al menos una vez.")

2026-07-28 23:51:29,731 [INFO] Cargando archivo: /home/ds/buckets/b1/datasets/preprocesado/preprocesado_grpProducto_fillNA_denseFull_tgtFilter.parquet

--- Dimensiones del DataFrame Final ---
Filas: 28080
Columnas: 14

--- Primeros 5 Registros ---


Agrupacion_ID,customer_id,product_id,periodo,tn,cust_request_qty,cust_request_tn,plan_precios_cuidados,cat1,cat2,cat3,brand,sku_size,descripcion
i64,i32,i64,date,f64,i64,f64,i64,str,str,str,str,i64,str
20242,0,20242,2019-12-01,4.03604,41,4.03604,0,"""FOODS""","""ADEREZOS""","""Ketchup""","""NATURA""",400,"""Ketchup tomate"""
20356,0,20356,2019-12-01,13.82404,181,13.82404,0,"""PC""","""CABELLO""","""SHAMPOO""","""SHAMPOO3""",350,"""Caspa"""
20148,0,20148,2019-12-01,57.16338,266,59.15709,0,"""HC""","""VAJILLA""","""Cristalino""","""LIMPIEX""",300,"""Lima"""
20720,0,20720,2019-12-01,8.95012,235,8.95012,0,"""PC""","""DEOS""","""Aero""","""NIVEA""",89,"""Aroma 1"""
20170,0,20170,2019-12-01,64.4108,268,64.4108,0,"""PC""","""CABELLO""","""ACONDICIONADOR""","""SHAMPOO2""",930,"""Palta"""



--- Nombres de los Campos (Columnas) ---
Agrupacion_ID
customer_id
product_id
periodo
tn
cust_request_qty
cust_request_tn
plan_precios_cuidados
cat1
cat2
cat3
brand
sku_size
descripcion
